# Automated LVEF Estimation: Spatiotemporal Baseline Model
### R(2+1)D-18 on Stanford EchoNet-Dynamic Dataset

This notebook trains and evaluates the baseline video regression model for continuous Left Ventricular Ejection Fraction (LVEF %) estimation from Apical-4-Chamber (A4C) echocardiograms.

**Hardware Requirement**: Kaggle GPU (NVIDIA P100 or T4 x2). Go to *Settings -> Accelerator -> GPU*.

In [ ]:
# 1. Verify GPU Acceleration and Environment
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    !nvidia-smi

In [ ]:
# 2. Clone GitHub Baseline Repository (or verify working directory)
import os
if not os.path.exists("src"):
    # Clone from user repository
    !git clone https://github.com/ThRavi95/BASELINE.git .
else:
    print("Baseline repository already present in working directory.")

In [ ]:
# 3. Verify EchoNet-Dynamic Dataset Presence
import glob
matches = glob.glob("/kaggle/input/**/FileList.csv", recursive=True)
if matches:
    print(f"[SUCCESS] EchoNet-Dynamic dataset found at: {os.path.dirname(matches[0])}")
else:
    print("[ACTION REQUIRED] Please add the EchoNet-Dynamic dataset to this Kaggle notebook via 'Add Input' on the right sidebar.")

In [ ]:
# 4. Launch Full Baseline Model Training & Official Test Evaluation
# Hyperparameters: 25 Epochs, Batch Size 16, AdamW (lr=1e-4), Mixed Precision (AMP)
!python kaggle/train_kaggle.py --epochs 25 --batch_size 16 --output_dir /kaggle/working/outputs

In [ ]:
# 5. Display Clinical Evaluation Results & Visualizations
from IPython.display import Image, display
import json

metrics_file = "/kaggle/working/outputs/test_metrics.json"
if os.path.exists(metrics_file):
    with open(metrics_file, "r") as f:
        m = json.load(f)
    print("="*50)
    print("OFFICIAL TEST SET CLINICAL EVALUATION (1,277 VIDEOS)")
    print("="*50)
    print(f"Mean Absolute Error (MAE):     {m['mae']:.2f}%")
    print(f"Root Mean Squared Error (RMSE):{m['rmse']:.2f}%")
    print(f"Pearson Correlation (r):       {m['pearson_r']:.3f}")
    print(f"Coefficient of Det. (R2):      {m['r2']:.3f}")
    print(f"Bland-Altman Bias:             {m['bland_altman_bias']:.2f}%")
    print(f"Limits of Agreement:           [{m['bland_altman_lower_loa']:.2f}%, {m['bland_altman_upper_loa']:.2f}%]")
    print(f"Low EF (<50%) Accuracy:        {m['binary_low_ef_accuracy']*100:.1f}%")
    print("="*50)

# Display generated figures
for fig_name in ["training_curves.png", "test_scatter.png", "test_bland_altman.png"]:
    fig_path = f"/kaggle/working/outputs/{fig_name}"
    if os.path.exists(fig_path):
        print(f"\nFigure: {fig_name}")
        display(Image(filename=fig_path))
